# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook explores the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset package using the `mlcroissant` library for programmatic access to metadata and records.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# The metadata property is an object, so print info using its attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Display all RecordSets in the dataset (each one has a unique @id)
record_sets = dataset.metadata.record_sets

print("Available Record Sets (by @id and name):\n")
for rs in record_sets:
    print(f"@id: {rs.id} | name: {getattr(rs, 'name', '')}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      @id: {field.id} | name: {getattr(field, 'name', '')}")
    print()

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Always reference entities using their `@id`.

In [ ]:
# Collect all record set @id's for extraction
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set @id: {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}\n")

if len(dataframes) > 0:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    print()
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter records, normalize numeric fields, group by categories, etc. All entity references use their `@id`.

In [ ]:
# Pick the first record set for EDA as an example

# Make sure to use the correct record set and field @id by inspecting the output of step 2 above.
record_set_id = list(dataframes.keys())[0]
df = dataframes[record_set_id]

print(f"Analysing record set: {record_set_id}")

# List all columns/fields
print("Columns (field @id):", df.columns.tolist())

# Try to get a numeric field automatically (assume float or int columns present)
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
# If not found, just pick the first column
if numeric_field is None:
    numeric_field = df.columns[0]
print(f"Using numeric field: {numeric_field}")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else None

# Filter records with value > mean (if possible)
if threshold is not None:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by another field if available (e.g., the second column)
    group_field = None
    for col in df.columns:
        if col != numeric_field:
            group_field = col
            break
    if group_field:
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print("Grouped data (mean):")
        display(grouped_df.head())
else:
    print(f"No numeric field found in {record_set_id} for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using `matplotlib`/`seaborn`. (All entity references remain via their column `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
if numeric_field and pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print(f"No numeric field available for visualization in {record_set_id}.")

## 6. Conclusion
In this notebook, we've loaded and explored the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the Croissant standard and `mlcroissant` library. We've programmatically loaded metadata, examined available record sets and fields by their `@id`, extracted data to pandas DataFrames, and run basic exploratory data analysis and visualizations. All dataset entities were referenced using their unique `@id`, enabling robust, reproducible access. For advanced analysis, further domain-specific processing and interpretation are recommended.